# Data Mining Assignment — Apache Spark Data Lake Preprocessing

DATA MINING ASSIGNMENT

SWETA KUMARI - LIT2023019

## 0. Install PySpark

In [1]:
!pip install -q pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


## 1. Create the Data Lake structure (Bronze / Silver)

In [2]:
import os, re, glob, shutil, hashlib

BASE       = "data_lake"
BRONZE     = f"{BASE}/bronze"
SILVER     = f"{BASE}/silver"
SILVER_OUT = f"{SILVER}/preprocessed_sales"
BRONZE_FILE = f"{BRONZE}/sales.csv"

os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER_OUT, exist_ok=True)

if not os.path.exists(BRONZE_FILE):
    try:
        from google.colab import files
        print("Please upload sales.csv ...")
        uploaded = files.upload()
        shutil.move(list(uploaded.keys())[0], BRONZE_FILE)
    except ImportError:
        if os.path.exists("sales.csv"):
            shutil.copy("sales.csv", BRONZE_FILE)
        else:
            raise FileNotFoundError("sales.csv not found - place it next to the notebook.")

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

BRONZE_MD5_BEFORE = md5(BRONZE_FILE)
print("Bronze file :", BRONZE_FILE)
print("MD5 (start) :", BRONZE_MD5_BEFORE)

Please upload sales.csv ...


Saving sales.csv to sales.csv
Bronze file : data_lake/bronze/sales.csv
MD5 (start) : 2b7d6737635600be42aa639140cd6ea9


## 2. Start Spark, load and inspect the raw data

In [3]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .master("local[*]")
         .appName("SalesDataLake")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

raw = (spark.read.option("header", True)
                 .option("inferSchema", False)
                 .option("quote", '"').option("escape", '"')
                 .csv(BRONZE_FILE))

print("Schema:")
raw.printSchema()
print("Sample rows:")
raw.show(5, truncate=False)
RAW_COUNT = raw.count()
print("Total records (raw):", RAW_COUNT)

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Sample rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------

### 2.1 Identify the important columns
Columns are detected by name (quantity, unit price, discount, dates). If a name is detected wrongly,
just overwrite the variables in this cell.

In [4]:
cols = raw.columns

def find(*patterns, exclude=None):
    for p in patterns:
        for c in cols:
            if re.search(p, c, re.I) and not (exclude and re.search(exclude, c, re.I)):
                return c
    return None

QTY_COL   = find(r"quantity", r"qty")
PRICE_COL = find(r"unit.?price", r"price", exclude=r"total")
DISC_COL  = find(r"discount")
DATE_COLS = [c for c in cols if re.search(r"date", c, re.I)]

ID_LIKE = r"(^|[_\s])(id|code|sku|email|phone|zip|postal)([_\s]|$)|id$"
NUMERIC_COLS = [c for c in [QTY_COL, PRICE_COL, DISC_COL] if c]

NULL_TOKENS = ["", "na", "n/a", "nan", "null", "none", "nil", "-", "?"]
def is_missing(c):
    return F.col(c).isNull() | F.lower(F.trim(F.col(c))).isin(NULL_TOKENS)

def to_num(c):
    """Strip currency symbols, commas, %, spaces and cast to double (non-numeric -> NULL)."""
    cleaned = F.regexp_replace(F.trim(F.col(c)), r"[^0-9.\-]", "")
    return F.when(cleaned == "", None).otherwise(cleaned.cast("double"))

other = [c for c in cols if c not in NUMERIC_COLS + DATE_COLS and not re.search(ID_LIKE, c, re.I)]
numeric_like = []
for c in other:
    row = raw.filter(~is_missing(c)).agg(
        F.count("*").alias("n"),
        F.sum(F.when(to_num(c).isNotNull(), 1).otherwise(0)).alias("ok")).first()
    if row["n"] and row["ok"] / row["n"] >= 0.9:
        numeric_like.append(c)
TEXT_COLS = [c for c in other if c not in numeric_like]

print("Quantity    :", QTY_COL)
print("Unit price  :", PRICE_COL)
print("Discount %  :", DISC_COL)
print("Date columns:", DATE_COLS)
print("Text columns:", TEXT_COLS)
print("Other numeric-like (kept as is):", numeric_like)
print("ID-like (kept as is):", [c for c in cols if re.search(ID_LIKE, c, re.I)])

Quantity    : quantity
Unit price  : unit_price
Discount %  : discount_percent
Date columns: ['order_date']
Text columns: ['customer_name', 'product', 'category', 'payment_method', 'city', 'state', 'order_status']
Other numeric-like (kept as is): []
ID-like (kept as is): ['order_id', 'customer_id']


### 2.2 Date parsing helper (several common formats are tried; day-first is preferred)

In [5]:
DATE_FORMATS = ["yyyy-M-d", "yyyy/M/d", "d-M-yyyy", "d/M/yyyy", "M-d-yyyy", "M/d/yyyy", "d.M.yyyy",
                "d MMM yyyy", "MMM d, yyyy", "d-MMM-yyyy", "d-MMM-yy", "yyyyMMdd"]
MIN_DATE, MAX_DATE = "2000-01-01", None

def parse_date(c):
    s = F.trim(F.col(c))
    d = F.coalesce(*[F.to_date(s, f) for f in DATE_FORMATS])
    upper = F.current_date() if MAX_DATE is None else F.lit(MAX_DATE).cast("date")
    return F.when((d >= F.lit(MIN_DATE).cast("date")) & (d <= upper), d)

from pyspark.sql import Window

def standardize_text(df, c):
    """Trim + collapse spaces, then make all spellings that differ only by case/spaces
    use the most common spelling (e.g. 'DELHI', 'delhi ' -> 'Delhi'; 'UPI' stays 'UPI')."""
    tmp = (df.withColumn("_v", F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
             .withColumn("_k", F.lower(F.col("_v"))))
    freq = tmp.filter(F.col("_v").isNotNull()).groupBy("_k", "_v").count()
    w = Window.partitionBy("_k").orderBy(F.desc("count"),
                                         (F.col("_v") == F.upper("_v")).cast("int"),
                                         (F.col("_v") == F.lower("_v")).cast("int"),
                                         F.col("_v"))
    canon = (freq.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
                 .select("_k", F.col("_v").alias("_canon")))
    out = tmp.join(canon, "_k", "left").withColumn(c, F.coalesce(F.col("_canon"), F.lit("Unknown")))
    return out.drop("_v", "_k", "_canon")

### 2.3 Data-quality profiling with Spark (all counts are computed, nothing is hard-coded)

In [6]:
# --- Duplicates ---------------------------------------------------------------
distinct_count = raw.dropDuplicates().count()
print("Exact duplicate rows :", RAW_COUNT - distinct_count)
print("Example duplicated records:")
raw.groupBy(cols).count().filter("count > 1").orderBy(F.desc("count")).show(5, truncate=False)

# --- Missing values (NULL, empty, 'NA', 'N/A', ...) ----------------------------
print("Missing values per column:")
raw.select([F.sum(F.when(is_missing(c), 1).otherwise(0)).alias(c) for c in cols]).show(truncate=False)

Exact duplicate rows : 10
Example duplicated records:
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100329  |C0151      |Varun Verma     |Bedside Table    |Furniture      |9       |3530      |20              |Net Banking     |Vijayawada|Andhra Pradesh|2025-04-08|Delivered   |2    |
|100055  |C0270      |Pranav Iyer     |Dry Fruits Pack  |Grocery        |7       |930       |12              |Cash on Delivery|Coimbatore|Tamil Nadu    |2026-08-20|Delivered   |2    |
|100317  |C0264      |Sidd

In [7]:
# --- Text-format inconsistencies ------------------------------------------------
collapse = lambda c: F.trim(F.regexp_replace(F.col(c), r"\s+", " "))
rows = []
for c in TEXT_COLS:
    r = raw.filter(~is_missing(c)).agg(
        F.countDistinct(c).alias("distinct_raw"),
        F.countDistinct(collapse(c)).alias("distinct_after_trim"),
        F.countDistinct(F.lower(collapse(c))).alias("distinct_ignoring_case"),
        F.sum(F.when(F.col(c) != collapse(c), 1).otherwise(0)).alias("rows_extra_spaces")).first()
    rows.append((c, r["distinct_raw"], r["distinct_after_trim"], r["distinct_ignoring_case"], r["rows_extra_spaces"]))
print("Text columns: distinct values before / after fixing spaces / after ignoring case")
spark.createDataFrame(rows, ["column", "distinct_raw", "distinct_after_trim", "distinct_ignoring_case", "rows_extra_spaces"]).show(truncate=False)

Text columns: distinct values before / after fixing spaces / after ignoring case
+--------------+------------+-------------------+----------------------+-----------------+
|column        |distinct_raw|distinct_after_trim|distinct_ignoring_case|rows_extra_spaces|
+--------------+------------+-------------------+----------------------+-----------------+
|customer_name |305         |302                |302                   |3                |
|product       |57          |57                 |57                    |0                |
|category      |13          |13                 |8                     |0                |
|payment_method|5           |5                  |5                     |0                |
|city          |18          |16                 |11                    |3                |
|state         |9           |9                  |9                     |0                |
|order_status  |5           |5                  |5                     |0                |
+--------

In [8]:
# --- Invalid numerical values --------------------------------------------------
checks = []
if QTY_COL:
    q = to_num(QTY_COL)
    checks += [(f"{QTY_COL}: non-numeric text", (~is_missing(QTY_COL)) & q.isNull()),
               (f"{QTY_COL}: zero / negative", q <= 0),
               (f"{QTY_COL}: not a whole number", q != F.floor(q))]
if PRICE_COL:
    p = to_num(PRICE_COL)
    checks += [(f"{PRICE_COL}: non-numeric text", (~is_missing(PRICE_COL)) & p.isNull()),
               (f"{PRICE_COL}: zero / negative", p <= 0)]
if DISC_COL:
    d = to_num(DISC_COL)
    checks += [(f"{DISC_COL}: non-numeric text", (~is_missing(DISC_COL)) & d.isNull()),
               (f"{DISC_COL}: outside 0-100", (d < 0) | (d > 100))]

res = raw.agg(*[F.sum(F.when(cond, 1).otherwise(0)).alias(name) for name, cond in checks]).first().asDict()
for k, v in res.items():
    print(f"{k:<45} {v}")

# --- Invalid / inconsistent dates ------------------------------------------------
for c in DATE_COLS:
    print(f"\nDate column: {c}")
    raw.agg(F.sum(F.when(is_missing(c), 1).otherwise(0)).alias("missing"),
            F.sum(F.when(~is_missing(c) & parse_date(c).isNull(), 1).otherwise(0)).alias("unparseable_or_out_of_range")).show()

    layout = F.regexp_replace(F.regexp_replace(F.trim(F.col(c)), r"[0-9]", "9"), r"[A-Za-z]", "A")
    raw.filter(~is_missing(c)).groupBy(layout.alias("date_layout")).count().orderBy(F.desc("count")).show(10, truncate=False)
    print("Examples of invalid dates:")
    raw.filter(~is_missing(c) & parse_date(c).isNull()).select(c).distinct().show(10, truncate=False)

quantity: non-numeric text                    0
quantity: zero / negative                     12
quantity: not a whole number                  0
unit_price: non-numeric text                  0
unit_price: zero / negative                   8
discount_percent: non-numeric text            0
discount_percent: outside 0-100               8

Date column: order_date
+-------+---------------------------+
|missing|unparseable_or_out_of_range|
+-------+---------------------------+
|      0|                          4|
+-------+---------------------------+

+-----------+-----+
|date_layout|count|
+-----------+-----+
|9999-99-99 |996  |
|99-99-9999 |1    |
|99/99/9999 |1    |
|AAA-A-AAAA |1    |
|9999/99/99 |1    |
+-----------+-----+

Examples of invalid dates:
+----------+
|order_date|
+----------+
|31/02/2026|
|not-a-date|
|2025-02-30|
|2026-13-05|
+----------+



## 3. Basic preprocessing → Silver layer
| # | Step | Method |
|---|------|--------|
| 1 | Duplicates | Remove **exact** duplicate rows |
| 2 | Missing values | Placeholders (`NA`, `N/A`, empty…) → NULL; text → `"Unknown"`; quantity / unit price → **median**; discount → `0` |
| 3 | Text | Trim, collapse repeated spaces, one consistent spelling per value (e.g. `DELHI` / `delhi ` → `Delhi`) |
| 4 | Validation | quantity must be a whole number > 0; unit price > 0; discount between 0 and 100. Invalid → NULL → imputed as above (record is kept) |
| 5 | Dates | Parsed from several layouts into `yyyy-MM-dd`; invalid / impossible / out-of-range dates → NULL (record is kept) |
| 6 | Preserve | No rows are dropped except exact duplicates |

In [9]:
df = raw.dropDuplicates()


for c in cols:
    df = df.withColumn(c, F.when(is_missing(c), None).otherwise(F.trim(F.col(c))))
for c in TEXT_COLS:
    df = standardize_text(df, c)
df = df.select(cols)

if QTY_COL:
    q = to_num(QTY_COL)
    df = df.withColumn(QTY_COL, F.when((q > 0) & (q == F.floor(q)), q))
if PRICE_COL:
    p = to_num(PRICE_COL)
    df = df.withColumn(PRICE_COL, F.when(p > 0, p))
if DISC_COL:
    d = to_num(DISC_COL)
    df = df.withColumn(DISC_COL, F.when((d >= 0) & (d <= 100), d))

medians = {}
for c in [QTY_COL, PRICE_COL]:
    if c:
        medians[c] = df.approxQuantile(c, [0.5], 0.001)[0]
        df = df.withColumn(c, F.coalesce(F.col(c), F.lit(medians[c])))
if QTY_COL:
    df = df.withColumn(QTY_COL, F.col(QTY_COL).cast("int"))
if DISC_COL:
    df = df.withColumn(DISC_COL, F.coalesce(F.col(DISC_COL), F.lit(0.0)))
print("Median values used for imputation:", medians)


for c in numeric_like:
    df = df.withColumn(c, to_num(c))

# 5. dates
for c in DATE_COLS:
    df = df.withColumn(c, F.date_format(parse_date(c), "yyyy-MM-dd"))

silver = df
SILVER_COUNT = silver.count()
print("Rows before:", RAW_COUNT, "| rows after:", SILVER_COUNT, "| removed:", RAW_COUNT - SILVER_COUNT)
silver.show(5, truncate=False)

Median values used for imputation: {'quantity': 5.0, 'unit_price': 1840.0}
Rows before: 1000 | rows after: 990 | removed: 10
+--------+-----------+-------------+-----------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|product    |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------------+-----------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100122  |C0188      |Ananya Sharma|Sugar 5kg  |Grocery        |6       |340.0     |10.0            |Debit Card      |Vijayawada|Andhra Pradesh|2026-06-28|Shipped     |
|100119  |C0191      |Aman Naidu   |Cricket Bat|Sports         |5       |2450.0    |12.0            |Net Banking     |Mumbai    |Maharashtra   |2025-07-15|Delivered   |
|100177  |C0072      |Arjun Sh

## 4. Save to the Silver layer

In [10]:
(silver.coalesce(1)
       .write.mode("overwrite").option("header", True)
       .csv(SILVER_OUT))

part = glob.glob(f"{SILVER_OUT}/part-*.csv")[0]
shutil.copy(part, f"{SILVER}/sales_silver.csv")

for p in sorted(glob.glob(f"{BASE}/**/*", recursive=True)):
    print(p)

data_lake/bronze
data_lake/bronze/sales.csv
data_lake/silver
data_lake/silver/preprocessed_sales
data_lake/silver/preprocessed_sales/_SUCCESS
data_lake/silver/preprocessed_sales/part-00000-248946d6-e7f4-4f08-a4d1-6385967aef66-c000.csv
data_lake/silver/sales_silver.csv


## 5. Verification

In [11]:

print(f"Row count BEFORE preprocessing (Bronze): {RAW_COUNT}")


silver_check = (spark.read.option("header", True).option("inferSchema", True).csv(SILVER_OUT))
print(f"Row count AFTER preprocessing (Silver) : {silver_check.count()}")
silver_check.printSchema()
silver_check.show(10, truncate=False)

print("Remaining NULLs per column in Silver:")
silver_check.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver_check.columns]).show(truncate=False)
print("Remaining exact duplicates in Silver:", silver_check.count() - silver_check.dropDuplicates().count())


print("Bronze unchanged:", md5(BRONZE_FILE) == BRONZE_MD5_BEFORE)

Row count BEFORE preprocessing (Bronze): 1000
Row count AFTER preprocessing (Silver) : 990
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+----------------------+---------------+--------+----------+----------------+----------------+-----+-----+----------+------------+
|order_id|customer_id|customer_name  |product               |category       |quantity|unit_price|discount_percent|payment_method  |city |state|order_date|order_status|
+--------+-----------+---